In [1]:
!pip install flask pyngrok flask-cors

In [2]:
!ngrok config add-authtoken 3BTiK0jP25TepA1V6RSl9jCmoBb_25AGFZihwm51RH3iPyaGM

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [3]:
pip install pandas scikit-learn nltk pymupdf openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 52.4 MB/s eta 0:00:00


In [4]:
import pandas as pd
import re
import fitz
import nltk
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)


excel_path = "/content/Model 1 Question Paper.csv"
data = pd.read_csv(excel_path)

excel_path = "/content/Model 2 Question Paper.csv"
data = pd.read_csv(excel_path)

data.columns = data.columns.str.strip().str.lower()

def find_column(possible_names):
    for name in possible_names:
        for col in data.columns:
            if name in col:
                return col
    return None

q_col = find_column(['question'])
a_col = find_column(['answer', 'key'])
m_col = find_column(['mark'])

if not q_col or not a_col or not m_col:
    raise Exception("Dataset must contain Question, Answer, and Marks columns")


def extract_text(file_path):
    if file_path.endswith(".pdf"):
        text = ""
        doc = fitz.open(file_path)
        for page in doc:
            text += page.get_text()
        doc.close()
        return text
    elif file_path.endswith(".txt"):
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()
    else:
        raise Exception("Only PDF or TXT supported")

student_file = "/content/full proper .txt"
student_text = extract_text(student_file)


def split_paragraphs(text):
    paras = re.split(r'\n+', text)
    return [p.strip() for p in paras if len(p.strip()) > 5]

student_paragraphs = split_paragraphs(student_text)


stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return " ".join(words)


def calculate_marks(similarity, max_marks):
    if similarity >= 0.8:
        return max_marks
    elif similarity >= 0.6:
        return round(max_marks * 0.75, 1)
    elif similarity >= 0.4:
        return round(max_marks * 0.5, 1)
    elif similarity >= 0.2:
        return round(max_marks * 0.25, 1)
    else:
        return 0


total_marks = 0
obtained_marks = 0

print("\n========== FULL EVALUATION ==========\n")

for index, row in data.iterrows():

    question = str(row[q_col])
    model_answer = clean_text(row[a_col])

    try:
        max_marks = float(row[m_col])
    except:
        max_marks = 1

    total_marks += max_marks

    best_similarity = 0
    best_answer = ""

    for para in student_paragraphs:
        clean_para = clean_text(para)

        if clean_para.strip() == "" or model_answer.strip() == "":
            continue

        try:
            vectorizer = TfidfVectorizer(ngram_range=(1,2))
            vectors = vectorizer.fit_transform([model_answer, clean_para])
            similarity = cosine_similarity(vectors[0], vectors[1])[0][0]
        except:
            similarity = 0

        if similarity > best_similarity:
            best_similarity = similarity
            best_answer = para

    if best_similarity < 0.2:
        marks = 0
        feedback = "Not Attempted"
        best_answer = "---"
    else:
        marks = calculate_marks(best_similarity, max_marks)
        feedback = "Correct" if marks == max_marks else "Partially Correct"

    obtained_marks += marks

    print(f"Q{index+1}: {question}")
    print(f"Matched Answer: {best_answer}")
    print(f"Similarity: {round(best_similarity, 2)}")
    print(f"Marks: {marks}/{max_marks}")
    print(f"Feedback: {feedback}")
    print("-"*50)


percentage = (obtained_marks / total_marks) * 100 if total_marks > 0 else 0

print("\n========== FINAL RESULT ==========")
print("Total Marks:", obtained_marks, "/", total_marks)
print("Percentage:", round(percentage, 2), "%")

if percentage >= 80:
    print("Overall: Excellent 🎉")
elif percentage >= 60:
    print("Overall: Good 👍")
elif percentage >= 40:
    print("Overall: Average 🙂")
else:
    print("Overall: Needs Improvement ⚠️")


========== FULL EVALUATION ==========

Q1: 1
Matched Answer: The best month to Visit Mussorie is September.
Similarity: 0.5
Marks: 0.5/1.0
Feedback: Partially Correct
--------------------------------------------------
Q2: 2
Matched Answer: 2) Both the teams Spent the Evening together as a party (3rd dinner)
Similarity: 0.56
Marks: 0.5/1.0
Feedback: Partially Correct
--------------------------------------------------
Q3: 3
Matched Answer: 3) Pierre was pinched the previous month because he behaved mischievo
Similarity: 0.61
Marks: 0.8/1.0
Feedback: Partially Correct
--------------------------------------------------
Q4: 4
Matched Answer: honest, hardworking, and humble in his actions.
Similarity: 0.35
Marks: 0.5/2.0
Feedback: Partially Correct
--------------------------------------------------
Q5: 5
Matched Answer: stayed near the dead male bird, and did not leave him, showing
Similarity: 0.52
Marks: 1.0/2.0
Feedback: Partially Correct
--------------------------------------------------

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok

app = Flask(__name__)
CORS(app)

@app.route('/evaluate', methods=['POST'])
def evaluate_api():
    file = request.files['file']

    # Save uploaded file
    file_path = "/content/full proper .txt"
    file.save(file_path)

    student_text = extract_text(file_path)
    student_paragraphs = split_paragraphs(student_text)

    total_marks = 0
    obtained_marks = 0

    for index, row in data.iterrows():
        model_answer = clean_text(row[a_col])

        try:
            max_marks = float(row[m_col])
        except:
            max_marks = 1

        total_marks += max_marks
        best_similarity = 0

        for para in student_paragraphs:
            clean_para = clean_text(para)

            if clean_para.strip() == "" or model_answer.strip() == "":
                continue

            try:
                vectorizer = TfidfVectorizer(ngram_range=(1,2))
                vectors = vectorizer.fit_transform([model_answer, clean_para])
                similarity = cosine_similarity(vectors[0], vectors[1])[0][0]
            except:
                similarity = 0

            if similarity > best_similarity:
                best_similarity = similarity

        marks = calculate_marks(best_similarity, max_marks)
        obtained_marks += marks

    # Percentage
    percentage = (obtained_marks / total_marks) * 100 if total_marks > 0 else 0

    # FIX: STORE feedback in variable (NOT print)
    if percentage >= 80:
        feedback = "Excellent"
    elif percentage >= 60:
        feedback = "Good"
    elif percentage >= 40:
        feedback = "Average"
    else:
        feedback = "Needs Improvement"

    #  Return response
    return jsonify({
        "marks": obtained_marks,
        "total": total_marks,
        "percentage": round(percentage, 2),
        "feedback": feedback
    })


#  ngrok
public_url = ngrok.connect(5000)
print("👉 COPY THIS URL:", public_url)

app.run(port=5000)

👉 COPY THIS URL: NgrokTunnel: "https://pyrenocarpic-domesticable-carma.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [21/May/2026 16:00:04] "POST /evaluate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/May/2026 16:11:52] "POST /evaluate HTTP/1.1" 200 -
